In [33]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import yaml

**Configuration and paths**


In [36]:
SEED = 42
VAL_RATIO = 0.20

CLASS_NAMES = {
    0: "D00_longitudinal_crack",
    1: "D10_transverse_crack",
    2: "D20_alligator_crack",
    3: "D40_pothole",
}

IMAGE_SUFFIXES = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
}

In [ ]:
PROJECT_ROOT = Path.cwd()

DATASET_ROOT = (PROJECT_ROOT / "data" / "processed")

ALL_IMAGES_DIR = (DATASET_ROOT / "all" / "images")
ALL_LABELS_DIR = (DATASET_ROOT / "all" / "labels")

TRAIN_IMAGES_DIR = (DATASET_ROOT / "train" / "images")
TRAIN_LABELS_DIR = (DATASET_ROOT / "train" / "labels")

VAL_IMAGES_DIR = (DATASET_ROOT / "val" / "images")
VAL_LABELS_DIR = (DATASET_ROOT / "val" / "labels")

CONFIGS_DIR = PROJECT_ROOT / "configs"
REPORTS_DIR = PROJECT_ROOT / "reports"

DATA_YAML_PATH = (CONFIGS_DIR / "rdd2022_india.yaml")
SPLIT_REPORT_PATH = (REPORTS_DIR / "rdd2022_split_manifest.json")

print("Unit root   :", PROJECT_ROOT.resolve())
print("Dataset root:", DATASET_ROOT.resolve())

Unit root   : /content
Dataset root: /content/data/processed


**Discover converted image-label pairs**

In [44]:
image_files = sorted(
    path for path in ALL_IMAGES_DIR.iterdir()
    if path.suffix.lower() in IMAGE_SUFFIXES
)

label_files = sorted(ALL_LABELS_DIR.glob("*.txt"))

images_by_stem = {path.stem: path for path in image_files}
labels_by_stem = {path.stem: path for path in label_files}

images_stems = set(images_by_stem)
label_stems = set(labels_by_stem)

print("Images:", len(image_files))
print("Labels:", len(label_files))

Images: 1934
Labels: 1934


In [45]:
images_without_labels = sorted(images_stems - label_stems)
labels_without_images = sorted(label_stems - images_stems)

if images_without_labels:
    print("Images without labels:",images_without_labels[:10])

if labels_without_images:
    print("Labels without images:", labels_without_images[:10])

**Read class information from labels**

In [46]:
def read_class_ids(label_path):
    text = label_path.read_text(encoding="utf-8").strip()
    class_ids = []
    if not text:
        return []
    
    for line_number, line in enumerate(text.splitlines(), start=1):
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(
                f"{label_path.name}, "
                f"line {line_number}: "
                "expected 5 values"
            )
        
        class_id = int(parts[0])
        if class_id not in CLASS_NAMES:
            raise ValueError(
                f"{label_path.name}: "
                f"invalid class ID {class_id}"
            )
            
        class_ids.append(class_id)

    return class_ids

In [47]:
classes_by_stem = {}

for stem in sorted(images_stems):
    classes_by_stem[stem] = read_class_ids(labels_by_stem[stem])